In [ ]:
#This just imports the stuff we will need to run the simulation

#main package: it imports some basic stuff as standard that is necessary to run a simulation.
#(look in jax_rmhd/__init__.py if you want to see what exactly is imported by default)
import jax_rmhd as jr
#jax version of numpy: this is recommended over the usual numpy to work with the just-in-time compilation, etc
import jax.numpy as jnp
# this imports all the Fourier transforms we will need
import jax.numpy.fft as ft
# plotting stuff
import matplotlib.pyplot as plt
#utilities
import os
from PIL import Image
import glob
import jax

import string
import random


#specifically importing stuff for snapshots and simple diagnostics
import jax_rmhd.snapshot_io as sn
import jax_rmhd.diagnostics as diag

#Q2: What do you think this function does specifically? Find it in the jax_rmhd folder, or create a new
#cell and run ??jr.init_cluster to read the code directly here.
#jr.init_cluster()

jax is using 32bit precision.


I'm going to make a function that creates random colors:

In [ ]:
def get_random_color(num):
    
    # hex_chars = string.digits + "ABCDEF"
    # hex_code = "".join(random.choices(hex_chars, k=6)) # joins all the random letters and numbers together
    # return f"#{hex_code}"

    global function_counter
    function_counter = function_counter + 1

    color_spectrum = ["red", "blue", "lime", "orange", "purple", "gold"]
    if function_counter > 6:
            random.shuffle(color_spectrum)
            function_counter = 0
    else:
          return color_spectrum[num]
            

Here is a function that uses a Gaussian to run a Ornstein-Uhlenbeck Process:

In [12]:
# Ornstein-Uhlenbeck Process
def ou_process(t_f, tcorr, delta_t, seed, rand_func, plot):
    # -- Create the keys to allow randomness --
    key = jax.random.PRNGKey(seed)
    key, f_key = jax.random.split(key)

    # -- Values to set up the Ornstein-Uhlenbeck Equation
    size = 1
    F_current = rand_func(f_key, shape = (size,))
    F_values = jnp.zeros((t_f + 1, size)) # initializes array and makes it empty
    F_values = F_values.at[0].set(F_current) # sets the initial value of the array to F_current
    
    # Graphing setup
    t_array = jnp.linspace(0, t_f, num = F_values.shape[0])
    legend_values = []
    if plot:
        plt.figure(figsize=(10, 5))
        plt.ylim(-5, 5)
        plt.xticks(jnp.arange(0.0, float(t_f), 2.5))
    
    # t_corr / step = number of graphs on plot
    step = 2
    colorint = 0
    for i in range(0, tcorr+2, step):
        for j in range(t_f):
            # -- Create the keys to allow randomness -- #
            key, f_prime_key = jax.random.split(key)
            F_prime = rand_func(f_prime_key, shape=(size, ))

            # Ornstein Uhlenbeck parameters
            theta = jnp.e**(-delta_t/(tcorr**(i)))
            F_next = theta*F_current + jnp.sqrt(1-theta**2) * F_prime # F_next = F(t+dt)

            # Building the next value off of the previous value
            F_current = F_next
            j = j + 1
            F_values = F_values.at[j].set(F_next)

        i = i + 1
        legend_values.append(f"{tcorr}^-{i}") # Puts all the values of tcorr into the legend


        if plot:
            plt.legend(legend_values)
            plt.plot(t_array, F_values, color=get_random_color(colorint))
            colorint = colorint + 1
            if colorint > 6:
                colorint = 0
        else:
            print("")

    
    if plot:
        plt.legend(legend_values)
        plt.show()
    else:
        print("")

Now we want to use the ou_process() function as our f values for our PDE:  
$$\frac{\partial u}{\partial t} + \textbf{u} \cdot \nabla u = -\nabla p + \upsilon \nabla^2\textbf{u} + \textbf{f}$$